In [1]:
print("ok")

ok


In [13]:
import os
os.chdir("../")

In [15]:
%pwd

'/home/sandip/sandip/python/GenAI/End-to-End-Medical-Chatbots'

In [14]:

from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [16]:
#Extrct Data from PDF

def load_pdf_file(data):
    loader=DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [17]:
extracted_data=load_pdf_file(data='data/')

In [19]:
#extracted_data

In [21]:
#Slpit the Data into text Chunks

def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.create_documents([doc.page_content for doc in extracted_data])
    return text_chunks

In [22]:
text_chunks=text_split(extracted_data)
print("Lenght of text chunks:", len(text_chunks))

Lenght of text chunks: 5859


In [23]:
from langchain.embeddings import HuggingFaceEmbeddings

In [25]:
#Download the Embedding from HuggingFace

def download_hugging_face_embedding():
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embedding

In [26]:
embedding=download_hugging_face_embedding()

/tmp/ipykernel_42586/3845606753.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [34]:
query_result=embedding.embed_query("What is the purpose of this trial?")
print("Lenght of query result:", len(query_result))

Lenght of query result: 384


In [49]:
from dotenv import load_dotenv
load_dotenv()


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPEN_AI_API_KEY = os.getenv("OPEN_AI_API_KEY")

#PINECONE_API_KEY

In [42]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)


{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-wmvetgr.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [50]:
import os

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY


In [44]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(documents = text_chunks, embedding=embedding, index_name=index_name)

In [45]:
#Existing Index

from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(embedding=embedding, index_name=index_name)

In [47]:
docsearch
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [48]:
retriever_docs = retriever.invoke("What is the Acne?")
retriever_docs

[Document(id='92339357-433d-4086-8f31-806d1d773471', metadata={}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='00d830b4-5773-4ee0-82f4-750d072dca8e', metadata={}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='fa44ad74-5327-441e-b399-24634a09b1ef', metadata={}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne,

In [53]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.4,max_tokens=500)

In [55]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = ("You are an assistant for quetion answering tasks."
                 "Use the follwing pieces of retrieved context to answer"
                 "the question. If you Don't know the answer, say you don't know, thnk you."
                 "Use three sentances maximum and keep the answer concise and to the point."
                 "\n\n"
                 "{context}"
                 )

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [56]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])


Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris, the medical term for common acne, affects nearly 17 million people in the United States.
